In [71]:
import os
import sys
import pickle


# Rutas internas del proyecto
from etl_pred import UserGenerator
from feature_engineer_pred import FeatureEngineer



In [72]:

# ============================================
# 📦 CONFIGURACIÓN DE RUTAS
# ============================================

# Ruta absoluta del proyecto raíz (sube tres niveles desde pred/)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
SRC_DIR = os.path.join(BASE_DIR, 'src')

# Agregar src al sys.path si no está
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

print(f"✅ sys.path incluye: {SRC_DIR}")

# ============================================
# 📂 IMPORTAR MÓDULOS INTERNOS
# ============================================

try:
    from etl_pred import UserGenerator
    from feature_engineer_pred import FeatureEngineer
    print("✅ Imports cargados correctamente.")
except ModuleNotFoundError as e:
    print("❌ Error al importar módulos:", e)

✅ sys.path incluye: c:\Users\Usuario\Proyecto_final_mlops\src\src
✅ Imports cargados correctamente.


In [73]:
 
# 🧠 FUNCIÓN PARA CARGAR EL MODELO
# ============================================

def get_model():
    # Ruta del modelo (donde realmente está)
    model_path = os.path.abspath(
    os.path.join(BASE_DIR, "app", "train", "models", "model_rf.pkl")
)

    print(f"📁 Buscando modelo en: {model_path}")

    # Validar si existe
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"❌ No se encontró el modelo en {model_path}")

    # Cargar modelo
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    print(f"✅ Modelo cargado correctamente desde: {model_path}")
    return model


In [74]:
def get_etl_data():
    # ============================================
    # 📂 Ruta del dataset (ajustada a tu estructura actual)
    # ============================================
    data_url = os.path.abspath(os.path.join("src", "app", "train", "data", "data_banknote_authentication.txt"))

    # ============================================
    # 🧩 Validar existencia del archivo
    # ============================================
    if not os.path.exists(data_url):
        raise FileNotFoundError(f"❌ No se encontró el dataset en: {data_url}")

    # ============================================
    # 🧠 Crear instancia del generador y cargar datos
    # ============================================
    user_generator = UserGenerator(url=data_url)
    df = user_generator.create_dataset()

    print(f"\n✅ Dataset cargado correctamente desde: {data_url}")
    print(f"📊 Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
    print(f"📋 Columnas: {list(df.columns)}")
    
    return df




In [75]:
import os, urllib.request

data_dir = "src/app/train/data"
os.makedirs(data_dir, exist_ok=True)

data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
save_path = os.path.join(data_dir, "data_banknote_authentication.txt")

urllib.request.urlretrieve(data_url, save_path)
print("✅ Dataset descargado en:", save_path)


✅ Dataset descargado en: src/app/train/data\data_banknote_authentication.txt


In [91]:
# ============================================
# 3️⃣ Ingeniería de features
# ============================================

import numpy as np
import pandas as pd

def get_data():
    df = get_etl_data()
    feature_engineer = FeatureEngineer(df)
    df = feature_engineer.create_features()

    # 🔹 Asegurar que las 3 columnas faltantes existan
    if 'abs_skewness' not in df.columns:
        df['abs_skewness'] = np.abs(df['skewness'])
    
    if 'var_entropy_ratio' not in df.columns:
        df['var_entropy_ratio'] = np.where(
            df['entropy'] != 0,
            df['variance'] / df['entropy'],
            0
        )

    if 'bucket_curtosis' not in df.columns:
        df['bucket_curtosis'] = pd.cut(
            df['curtosis'],
            bins=3,
            labels=['low', 'medium', 'high']
        )

    print("✅ Features generadas correctamente. Total columnas:", len(df.columns))
    print("📋 Columnas finales:", list(df.columns))
    return df



In [92]:
# ============================================
# 4️⃣ Predicción
# ============================================

def predict(model, df):
    prediction = model.predict(df)
    print("✅ Predicciones generadas correctamente.")
    return prediction

In [93]:
# ============================================
# 5️⃣ Guardar predicciones
# ============================================

def save_prediction(df, prediction):
    """
    Guarda las predicciones generadas en un archivo CSV dentro del módulo pred-batch.
    """
    # Ruta ajustada a tu estructura actual
    output_dir = os.path.join("src", "app", "pred-batch", "queries", "predictions")
    os.makedirs(output_dir, exist_ok=True)
    
    # Agregar las predicciones al DataFrame
    df["prediction"] = prediction
    df["prediction"] = df["prediction"].astype(int)
    df["prediction"] = df["prediction"].map({0: "No", 1: "Sí"})
    
    # Guardar el archivo CSV
    output_path = os.path.join(output_dir, "predictions.csv")
    df.to_csv(output_path, index=False)
    
    print(f"✅ Predicciones guardadas en: {output_path}")
    return df

In [94]:
model = get_model()


📁 Buscando modelo en: c:\Users\Usuario\Proyecto_final_mlops\src\app\train\models\model_rf.pkl
✅ Modelo cargado correctamente desde: c:\Users\Usuario\Proyecto_final_mlops\src\app\train\models\model_rf.pkl


In [95]:
model

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [96]:
df = get_data()



🚀 Generando dataset de autenticación de billetes...


📂 Cargando dataset de billetes...
✅ Dataset cargado correctamente: 1372 filas, 5 columnas.

🧹 Revisando datos nulos...
variance    0
skewness    0
curtosis    0
entropy     0
class       0
dtype: int64
✅ No se encontraron valores nulos.

📊 Distribución de la clase (0=auténtico, 1=falso):
class
0    762
1    610
Name: count, dtype: int64

📈 Proporción:
class
0    0.555
1    0.445
Name: proportion, dtype: float64

✅ Dataset final listo para procesamiento.

✅ Dataset cargado correctamente desde: c:\Users\Usuario\Proyecto_final_mlops\src\app\pred-batch\queries\src\app\train\data\data_banknote_authentication.txt
📊 Filas: 1372 | Columnas: 5
📋 Columnas: ['variance', 'skewness', 'curtosis', 'entropy', 'class']
🧠 Generando nuevas características del dataset...
✅ Nuevas columnas creadas correctamente:
['variance', 'skewness', 'curtosis', 'entropy', 'class', 'magnitude', 'curtosis_minus_skewness', 'entropy_to_curtosis_ratio', 'skewness_to_var

In [97]:
df


,variance,skewness,curtosis,entropy,class,magnitude,curtosis_minus_skewness,entropy_to_curtosis_ratio,skewness_to_variance_ratio,abs_entropy,abs_skewness,var_entropy_ratio,bucket_curtosis
0,3.62160,8.66610,-2.8073,-0.44699,0,9.813155,-11.47340,0.159224,2.392893,0.44699,8.66610,-8.102195,low
1,4.54590,8.16740,-2.4586,-1.46210,0,9.775177,-10.62600,0.594688,1.796652,1.46210,8.16740,-3.109158,low
2,3.86600,-2.63830,1.9242,0.10645,0,5.061666,4.56250,0.055322,-0.682437,0.10645,2.63830,36.317520,low
3,3.45660,9.52280,-4.0112,-3.59440,0,11.473502,-13.53400,0.896091,2.754962,3.59440,9.52280,-0.961663,low
4,0.32924,-4.45520,4.5718,-0.98880,0,6.468098,9.02700,-0.216282,-13.531770,0.98880,4.45520,-0.332969,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367,0.40614,1.34920,-1.4501,-0.55949,1,2.097882,-2.79930,0.385829,3.322007,0.55949,1.34920,-0.725911,low
1368,-1.38870,-4.87730,6.4774,0.34179,1,8.233473,11.35470,0.052767,3.512134,0.34179,4.87730,-4.063021,medium
1369,-3.75030,-13.45860,17.5932,-2.77710,1,22.636953,31.05180,-0.157851,3.588673,2.77710,13.45860,1.350438,high
1370,-3.56370,-8.38270,12.3930,-1.28230,1,15.433741,20.77570,-0.103470,2.352246,1.28230,8.38270,2.779147,high


In [98]:
prediction = predict(model, df)

✅ Predicciones generadas correctamente.


In [99]:
prediction

array([0, 0, 0, ..., 1, 1, 1], shape=(1372,))

In [100]:
print(prediction[:1000])



[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 

In [101]:
save_prediction(df, prediction)


✅ Predicciones guardadas en: src\app\pred-batch\queries\predictions\predictions.csv


,variance,skewness,curtosis,entropy,class,magnitude,curtosis_minus_skewness,entropy_to_curtosis_ratio,skewness_to_variance_ratio,abs_entropy,abs_skewness,var_entropy_ratio,bucket_curtosis,prediction
0,3.62160,8.66610,-2.8073,-0.44699,0,9.813155,-11.47340,0.159224,2.392893,0.44699,8.66610,-8.102195,low,No
1,4.54590,8.16740,-2.4586,-1.46210,0,9.775177,-10.62600,0.594688,1.796652,1.46210,8.16740,-3.109158,low,No
2,3.86600,-2.63830,1.9242,0.10645,0,5.061666,4.56250,0.055322,-0.682437,0.10645,2.63830,36.317520,low,No
3,3.45660,9.52280,-4.0112,-3.59440,0,11.473502,-13.53400,0.896091,2.754962,3.59440,9.52280,-0.961663,low,No
4,0.32924,-4.45520,4.5718,-0.98880,0,6.468098,9.02700,-0.216282,-13.531770,0.98880,4.45520,-0.332969,medium,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367,0.40614,1.34920,-1.4501,-0.55949,1,2.097882,-2.79930,0.385829,3.322007,0.55949,1.34920,-0.725911,low,Sí
1368,-1.38870,-4.87730,6.4774,0.34179,1,8.233473,11.35470,0.052767,3.512134,0.34179,4.87730,-4.063021,medium,Sí
1369,-3.75030,-13.45860,17.5932,-2.77710,1,22.636953,31.05180,-0.157851,3.588673,2.77710,13.45860,1.350438,high,Sí
1370,-3.56370,-8.38270,12.3930,-1.28230,1,15.433741,20.77570,-0.103470,2.352246,1.28230,8.38270,2.779147,high,Sí
